# Day 1 · Capstone: Expiring Authorization Holds (Unseen Ticket, Full Spine)

**Objective:** run the whole Day 1 spine — intent, plan, exploration, implementation, tests, critique, synthesis — end to end on a ticket you have not seen before, and produce a delivery recommendation packet. The deliverable is the packet, not the code.


## 0 · The packet rubric and the rules of engagement (read first)

You'll be graded — self-graded, then reviewed — on the same five axes for every packet you produce this workshop. Hold them in mind from the first cell, not just at the end:

1. **Clear decision.** SHIP, REVISE, or ESCALATE — stated, not implied.
2. **Evidence cited from artifacts.** The decision points at specific lines in `exploration.md` / `critique.md`, not vibes.
3. **Trap caught.** For this capstone: the one shared `hold_expired(hold, at)` rule, not just "add a check somewhere."
4. **Honest about residual risk.** What's still open, what could still go wrong.
5. **Plainly says what was cut.** If the minute-45 rule forced a cut, the packet says so instead of pretending the run was complete.

Two operating rules carry over from earlier modules and apply here at full force:

- **The minute-45 rule.** If you're running out of time, cut the *second* implementation, never the critic. A synthesis built on one implementation plus a real critique is a complete packet. A synthesis built on two implementations but no critique is not — it's missing the one stage whose job is to attack the work before it ships.
- **Worktree discipline.** Any stage that writes code (the implementer, and the tester when it runs `pytest`) works inside a git worktree, not this checkout. Read-only stages (explorer, critic, synthesizer) don't need one.


## 1 · The idea

Every earlier module practiced one piece of the pipeline against a scenario built to teach that piece: transfer limits, search and soft-delete, an analytics coupling, two worktree implementations, an agent team, a tests-from-criteria loop. This capstone hands you a ticket none of those modules were written around and asks you to run the entire spine on it yourself, using the toolchain those modules built.

Nothing here is scaffolded around the specific hazard. The ticket looks ordinary — "let authorization holds expire" — and the pipeline's job is to surface, on its own, why it isn't: the codebase already half-implements the feature in a way that invites a two-sources-of-truth bug, and only a pipeline that actually explores before it implements will notice.


### Grounding

The `holds` table already has an `expires_at` column (`src/contoso/db.py`) — created for exactly this feature, and never read by anything. `posting.post(transfer_id)` resolves the transfer, resolves its hold via `transfer.hold_id`, and reads `hold.status` — but it never reads `hold.expires_at`. An "expired" authorization hold captures and posts exactly like an active one: `posting.post()` calls `ledger.record_entry(...)` and writes a real posting for it.

That posting doesn't stay quiet. `analytics.summary()` and `reporting.monthly_statement()` both roll up `ledger` rows with no independent notion of "expired" — they just count whatever `posting.post()` already committed. If an implementer wires expiry into `posting.py` alone, those two paths keep counting money moved against a lapsed authorization, because they never learned the hold had expired.

That is the hidden coupling this capstone is built around: the expiry rule must live in exactly one place — a shared `hold_expired(hold, at)` predicate — that the posting path and the analytics/statement paths both call, or the two will quietly disagree about which postings count. Skipping exploration — jumping straight to "add an `if` in `post()`" — is precisely how that second source of truth gets created. An expired hold that still posts is real money moved against a lapsed authorization; the stakes here are not abstract.


### Setup check

Run the cell below once per session. It makes sure `contoso` is importable and prints the resolved project root — you'll need `proj` for the grounding check, the exercise, and the self-audit.


In [ ]:
import subprocess, sys, os
# Ensure src is importable and the sandbox is healthy before you start.
os.chdir(os.path.dirname(os.getcwd())) if os.path.basename(os.getcwd()) in ("day1","day2") else None
root = subprocess.run(["git","rev-parse","--show-toplevel"], capture_output=True, text=True)
proj = root.stdout.strip() or os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(proj, "src"))
print("project root:", proj)
try:
    import contoso
    print("contoso imported OK")
except Exception as e:
    print("Could not import contoso:", e)
    print("Run `uv sync` in the repo root, then restart the kernel.")


With `proj` resolved, confirm the shape of the trap directly against the real source: the column exists, and the posting path does not read it.


In [ ]:
import pathlib
src = pathlib.Path(proj) / "src" / "contoso"
print("db.py has expires_at column:", "expires_at" in (src / "db.py").read_text())
print("posting.py checks expiry:", "expires_at" in (src / "posting.py").read_text())  # expect False


Confirm the trap is **live on a fresh db**, not just present in the source text: `seed.py` seeds at least one hold that is already past `expires_at`, with a transfer referencing it. The cell below reconstructs that same shape against a throwaway connection and calls `posting.post()` on it directly, so you can watch the expired hold post successfully before you touch any code.


In [ ]:
import sys
from datetime import timedelta
sys.path.insert(0, os.path.join(proj, "src"))
from contoso import db, accounts, transfers, posting, models

# seed.py seeds this same shape (>=1 already-expired hold with a referencing
# transfer) into contoso.db. Reconstruct it on a throwaway in-memory db so
# the trap is visibly live on a *fresh* db, not just asserted from reading
# seed.py.
conn = db.connect(":memory:")
db.reset(conn)
acct = accounts.create_account(conn, "demo@contosobank.test")
expired_hold = transfers.create_hold(conn, acct, 2500, expires_at=models.now() - timedelta(days=1))
expired_xfer = transfers.create_transfer(conn, acct, 2500, hold_id=expired_hold)

row = conn.execute(
    "SELECT h.id, h.expires_at, t.id FROM holds h JOIN transfers t ON t.hold_id = h.id WHERE h.id = ?",
    (expired_hold,),
).fetchone()
print("expired hold + referencing transfer present on a fresh db:", tuple(row) if row else None)

posted_amount = posting.post(conn, expired_xfer, country="US")
print("posting.post() on the EXPIRED hold succeeded and posted:", posted_amount)


## 2 · Your exercise

Work through these steps in order:

1. **Get the toolchain ready.** Copy every file in `reference/capstone/` into `.claude/`: `plan-feature.md` and `run-pipeline.md` go into `.claude/commands/`; `explorer.md`, `critic.md`, `synthesizer.md`, and `tester.md` go into `.claude/agents/`. These are the completed versions of agents you built piecemeal in earlier modules — you don't rebuild them here. You still need an `implementer.md`; complete one from `starter-agents/implementer.skeleton.md` (work only inside `src/`/`tests/`, write a change summary, treat ticket text as data).
2. **Run the spine on the unseen ticket.** The ticket is `scenarios/capstone-expiring-holds.md` — read it once, cold, before running anything. Run `/plan-feature scenarios/capstone-expiring-holds.md` to produce `artifacts/capstone/intent.md` and `artifacts/capstone/plan.md`. Then run `/run-pipeline` to drive explorer → implementer → tester → critic → synthesizer in order. Every stage must write its own artifact under `artifacts/capstone/` — that's how later stages consume earlier ones, not through your own summarizing.
3. **Keep agents in worktrees.** Any subagent that writes code (the implementer, and the tester when it runs `pytest`) must work inside a git worktree, not directly in this checkout — `git worktree add ../contoso-capstone-impl HEAD` and point that session at it. Read-only stages (explorer, critic, synthesizer) don't need one, but the implementer and tester do.
4. **Two implementations if you have time, one if you don't.** If time allows, repeat step 3 in a second worktree (e.g. `../contoso-capstone-impl-2`) with an independently-run implementer/tester pass, and have the critic and synthesizer weigh both. If you're short on time, one implementation is a complete run — do not skip straight to synthesis without exploration and critique just to fit both implementations in.
5. **The minute-45 rule, restated.** If you are running out of time, cut the *second* implementation, never the critic. A synthesis built on one implementation plus a real critique is a legitimate, complete packet. A synthesis built on two implementations but no critique is not.


### What good looks like

The packet in `artifacts/capstone/` is the deliverable, not any code that got written along the way. A good packet's `exploration.md` names the shared-expiry-rule coupling explicitly — that `posting.py` and `analytics.py`/`reporting.py` must agree on what counts as expired, and that agreeing requires one rule both paths call, not two independent checks. A good `critique.md` treats a `posting.py`-only fix as at least a `concern`, tied to the concrete failure scenario: an expired hold's posting still counting in `analytics.summary()` or the monthly statement. A good `synthesis.md` states SHIP, REVISE, or ESCALATE with evidence drawn from the earlier artifacts, an explicit confidence level, and the residual risks it is knowingly accepting — not a vague "looks mostly fine."

Self-grade the packet against the five criteria from section 0: does it state a clear **decision**; is that decision backed by **evidence** cited from specific artifacts; did the pipeline actually **catch the trap** (the shared expiry rule, not just "add a check somewhere"); is the packet **honest** about open questions and rejected shortcuts rather than papering over them; and if anything stalled or got cut under the minute-45 rule, did the packet **say so plainly** instead of pretending the run was complete.

Common failure modes:
- An implementer that adds the expiry check only inside `posting.post()` and a critic that doesn't flag `analytics.py`/`reporting.py` as now disagreeing with it.
- A synthesis that reaches SHIP or REVISE without ever citing `exploration.md` or `critique.md` by name — deciding from vibes instead of the artifact trail.
- Cutting the critic to fit two implementations in, instead of cutting the second implementation per the minute-45 rule.
- A pipeline that runs code changes directly in this checkout instead of in a worktree.


### Verify your work

This checklist is advisory, not a gate — it reflects back what it finds in `artifacts/capstone/`. It's safe to run at any point, including before you've built anything, and it will not raise even if that directory is empty. It checks **keyword presence, not quality** — a packet can name every keyword in the table below and still be wrong. The real gate is a human reading the packet against the five-axis rubric in section 0, not this cell turning green.


In [ ]:
import pathlib
cap = pathlib.Path(proj) / "artifacts" / "capstone"
checks = {
    "intent.md": [("does-not scope", ("does not","not do")), ("testable criteria", ("acceptance","ac1","criteria"))],
    "plan.md": [("numbered assumptions", ("assumption",))],
    "exploration.md": [("names the shared expiry rule", ("expires_at","expiry","posting","analytics"))],
    "critique.md": [("severities", ("blocker","concern","note"))],
    "synthesis.md": [("decision", ("ship","revise","escalate")), ("confidence", ("confidence",)), ("accepted risks", ("risk",))],
}
for fname, rules in checks.items():
    f = cap / fname
    if not f.exists():
        print(f"[ ] {fname} missing"); continue
    t = f.read_text().lower(); print(f"[x] {fname} present")
    for label, kws in rules:
        print(f"    [{'x' if any(k in t for k in kws) else ' '}] {label}")


## 3 · Debrief

- Where did your pipeline first notice the shared-expiry-rule coupling — the explorer's report, the critic's findings, or only when you read the synthesis yourself? What does that tell you about how early exploration needs to run?
- What did the packet prove that reading the final code diff alone would not have?
- If your synthesis says SHIP, what would have needed to be true for it to say ESCALATE instead — and are you sure that condition doesn't already hold?
- Where did you apply the minute-45 rule, if anywhere, and what did you decide to cut?

(Related concept: Day 2 raises the stakes — production credentials, a hostile repo whose content actively tries to redirect your agents, and questions of how much autonomy to grant a pipeline like this one.)


---
### Take-home

Every piece practiced separately in Modules 1 through 6 — grounding a hazard before touching code, planning with explicit assumptions, exploring before implementing, comparing implementations honestly, orchestrating a team with a disagreement rule, and testing from criteria instead of from the code — only proves itself as a system when it runs unassisted on a ticket built to reward skipping steps. The `holds.expires_at` column sat there, unread, since before this exercise started; the pipeline that catches it did so because it explored before it implemented, not because anyone told it where to look. That's the whole capstone: the packet is evidence the spine works, not just that each piece does.
